**Autores:**

- Gonçalo Henriques (123422)
- Rodrigo Sousa (123390)

In [1]:
import os
os.chdir('/home/jovyan/work')  # garante que os caminhos relativos partem da raiz do projeto

from pymongo import MongoClient, UpdateOne
from datetime import datetime, timezone
import pprint

In [2]:
# Dentro do Docker, o container MongoDB é acessível pelo nome do serviço
client = MongoClient("mongodb://mongodb:27017/")
db = client["london_bikes"]
client.admin.command("ping")
client

MongoClient(host=['mongodb:27017'], document_class=dict, tz_aware=False, connect=True)

---
# **3 - Estações: Análise e Limpeza**

Este notebook analisa e limpa a collection `stations_raw` do MongoDB, produzida pelo `1-DataIngestion.ipynb`.

**Objetivos:**
1. Identificar problemas de qualidade de dados em `stations_raw`
2. Remover duplicados — ficar com um documento por estação (snapshot mais recente)
3. Converter tipos de dados: strings booleanas → `bool`, timestamps Unix em ms → `ISODate`
4. Normalizar nomes das estações (artefactos de espaçamento)
5. Derivar o campo `is_active` (estação sem data de desativação)
6. Enriquecer com coordenadas `lat`/`lon` a partir das collections de bicicletas
7. Escrever o resultado na collection `stations_clean`

A collection limpa servirá como entrada de características por estação para o clustering em `4-Modelling.ipynb`.

**NOTA:** Este notebook deve ser executado após `1-DataIngestion.ipynb` e `2-DataUnderstanding.ipynb`.

---
## **1. Inspeção de `stations_raw`**

In [3]:
stations_raw = db["stations_raw"]

total_docs = stations_raw.count_documents({})
print(f"Total de documentos em stations_raw: {total_docs:,}")

print("\nDocumento de exemplo:")
pprint.pprint(stations_raw.find_one())

Total de documentos em stations_raw: 385,731

Documento de exemplo:
{'_id': ObjectId('6a0cb33c4525380ceb8d4ce3'),
 'common_name': 'River Street , Clerkenwell',
 'install_date': '1278947280000',
 'installed': 'true',
 'locked': 'false',
 'place_id': 'BikePoints_1',
 'removal_date': None,
 'snapshot_date': '2022-05-27',
 'temporary': 'false',
 'terminal_name': '001023'}


### **1.1 Análise de Valores Nulos e Ausentes**

In [4]:
fields = [
    "place_id", "common_name", "installed", "temporary",
    "install_date", "terminal_name", "locked", "removal_date", "snapshot_date"
]

print("=== Contagem de Nulos / Valores Ausentes ===")
for field in fields:
    n = stations_raw.count_documents({
        "$or": [{field: None}, {field: {"$exists": False}}, {field: ""}, {field: "NULL"}]
    })
    nota = "(esperado — estações ativas)" if field == "removal_date" and n > 0 else ""
    print(f"  {field:<20}: {n:>8,}  ({n/total_docs*100:5.1f}%)  {nota}")

=== Contagem de Nulos / Valores Ausentes ===
  place_id            :        0  (  0.0%)  
  common_name         :        0  (  0.0%)  
  installed           :        0  (  0.0%)  
  temporary           :        0  (  0.0%)  
  install_date        :   41,043  ( 10.6%)  
  terminal_name       :        0  (  0.0%)  
  locked              :       16  (  0.0%)  
  removal_date        :  384,293  ( 99.6%)  (esperado — estações ativas)
  snapshot_date       :        0  (  0.0%)  


### **1.2 Remoção de Duplicados: Snapshots por Estação**

Cada ficheiro CSV de estações (um por data) foi ingerido separadamente, produzindo um documento por estação por data de snapshot. A collection `stations_raw` contém assim o histórico completo de alterações ao nível da estação. Para a fase de modelação é necessário um único registo atual por estação.

In [5]:
estacoes_distintas = len(stations_raw.distinct("place_id"))
print(f"Estações distintas (place_id):  {estacoes_distintas:,}")
print(f"Total de documentos:            {total_docs:,}")
print(f"Média de snapshots por estação: {total_docs / estacoes_distintas:.1f}")

resultado = list(stations_raw.aggregate([
    {"$group": {"_id": "$place_id", "count": {"$sum": 1}}},
    {"$group": {"_id": None,
        "min_snapshots": {"$min": "$count"},
        "max_snapshots": {"$max": "$count"},
        "avg_snapshots": {"$avg": "$count"}
    }}
]))
print("\nDistribuição de snapshots:")
pprint.pprint(resultado[0])

Estações distintas (place_id):  812
Total de documentos:            385,731
Média de snapshots por estação: 475.0

Distribuição de snapshots:
{'_id': None,
 'avg_snapshots': 475.0381773399015,
 'max_snapshots': 486,
 'min_snapshots': 1}


### **1.3 Problemas de Tipos de Dados**

Todos os campos foram ingeridos como strings (`inferSchema=false`). Três categorias de problemas de tipo necessitam de correção:
- Strings booleanas (`"true"` / `"false"`)
- Strings de timestamp Unix em milissegundos (ex: `"1279026060000"`)
- Strings de data para `snapshot_date` (ex: `"2022-05-20"`)

In [6]:
# Confirmar valores distintos nos campos booleanos
for field in ["installed", "locked", "temporary"]:
    print(f"Valores distintos em '{field}': {stations_raw.distinct(field)}")

Valores distintos em 'installed': ['false', 'true']
Valores distintos em 'locked': [None, 'false', 'true']
Valores distintos em 'temporary': ['false', 'true']


In [7]:
# Amostrar valores de install_date e converter para verificar o formato (Unix ms → data)
amostra = list(stations_raw.aggregate([
    {"$match": {"install_date": {"$nin": [None, "", "NULL"]}}},
    {"$project": {"_id": 0, "place_id": 1, "install_date": 1}},
    {"$limit": 6}
]))

print("Valores de install_date e respetivos equivalentes convertidos:")
for doc in amostra:
    try:
        dt = datetime.fromtimestamp(int(doc["install_date"]) / 1000, tz=timezone.utc)
        print(f"  {doc['place_id']}: '{doc['install_date']}' → {dt.strftime('%Y-%m-%d')}")
    except (ValueError, TypeError) as e:
        print(f"  {doc['place_id']}: '{doc['install_date']}' → ERRO: {e}")

Valores de install_date e respetivos equivalentes convertidos:
  BikePoints_1: '1278947280000' → 2010-07-12
  BikePoints_2: '1278585780000' → 2010-07-08
  BikePoints_3: '1278240360000' → 2010-07-04
  BikePoints_4: '1278241080000' → 2010-07-04
  BikePoints_5: '1278241440000' → 2010-07-04
  BikePoints_6: '1278241680000' → 2010-07-04


### **1.4 Normalização dos Nomes das Estações**

In [8]:
# Espaços no início/fim
espacos_ext = stations_raw.count_documents({"common_name": {"$regex": "^\\s|\\s$"}})
print(f"Nomes com espaço no início/fim: {espacos_ext:,}")

# Artefacto espaço-antes-vírgula (ex: "Bayley Street , Bloomsbury")
espaco_virgula = list(stations_raw.aggregate([
    {"$match": {"common_name": {"$regex": " ,"}}},
    {"$group": {"_id": "$common_name"}},
    {"$limit": 10}
]))
print(f"\nNomes com artefacto ' ,' (espaço antes de vírgula): {len(espaco_virgula)} valores distintos")
for doc in espaco_virgula:
    print(f"  '{doc['_id']}'")

Nomes com espaço no início/fim: 0

Nomes com artefacto ' ,' (espaço antes de vírgula): 10 valores distintos
  'Eversholt Street , Camden Town'
  'Northington Street , Holborn'
  'High Holborn , Covent Garden'
  'River Street , Clerkenwell'
  'Tower Gardens , Tower'
  'Montserrat Road , Putney'
  'Kennington Road  , Vauxhall'
  'Earnshaw Street , Covent Garden'
  'Princedale Road , Holland Park'
  'Regent's Row , Haggerston'


### **1.5 Análise das Datas de Remoção**

In [9]:
removidas = stations_raw.count_documents({
    "$and": [
        {"removal_date": {"$ne": None}},
        {"removal_date": {"$ne": ""}},
        {"removal_date": {"$ne": "NULL"}}
    ]
})
print(f"Registos com removal_date (desativadas): {removidas:,}")
print(f"Registos sem removal_date (ativas):      {total_docs - removidas:,}")

amostra_removidas = list(stations_raw.aggregate([
    {"$match": {"removal_date": {"$nin": [None, "", "NULL"]}}},
    {"$project": {"_id": 0, "place_id": 1, "common_name": 1, "removal_date": 1}},
    {"$limit": 5}
]))
if amostra_removidas:
    print("\nExemplos de estações desativadas:")
    for doc in amostra_removidas:
        dt = datetime.fromtimestamp(int(doc["removal_date"]) / 1000, tz=timezone.utc)
        print(f"  {doc['place_id']}: '{doc['common_name']}' removida em {dt.strftime('%Y-%m-%d')}")

Registos com removal_date (desativadas): 1,438
Registos sem removal_date (ativas):      384,293

Exemplos de estações desativadas:
  BikePoints_391: 'Clifford Street, Mayfair' removida em 2018-10-18
  BikePoints_705: 'Upper Richmond Road, Putney' removida em 2018-10-18
  BikePoints_316: 'Warwick Row, Westminster' removida em 2018-10-18
  BikePoints_391: 'Clifford Street, Mayfair' removida em 2018-10-18
  BikePoints_705: 'Upper Richmond Road, Putney' removida em 2018-10-18


---
## **2. Resumo dos Problemas de Qualidade de Dados**

| Problema | Detalhe | Correção |
|---|---|---|
| **Duplicados por estação** | ~486 registos por estação | Ficar só com o mais recente |
| **Strings booleanas** | `installed`, `locked`, `temporary` como `"true"`/`"false"` | `$cond` → `bool` real |
| **Strings de timestamp Unix ms** | `install_date` / `removal_date` como strings ms (ex: `"1279026060000"`) | `$toDate($toLong(...))` |
| **String de data** | `snapshot_date` como `"YYYY-MM-DD"` | `$dateFromString` |
| **Espaço-antes-vírgula** | Alguns `common_name` contêm artefacto `" ,"` (incluindo duplo espaço) | `$replaceAll` encadeado + `$trim` |
| **Sem coordenadas** | Estações sem `lat`/`lon` nesta collection | Enriquecer das collections de bicicletas |

---
## **3. Pipeline de Limpeza → `stations_clean`**

Um único pipeline de agregação MongoDB trata todos os passos de limpeza e escreve o resultado na collection `stations_clean` via `$out`.

In [10]:
pipeline = [
    # 1. Ordenar por snapshot_date decrescente → snapshot mais recente fica primeiro
    {"$sort": {"snapshot_date": -1}},

    # 2. Remover duplicados: agrupar por place_id, manter os valores do snapshot mais recente
    {"$group": {
        "_id":             "$place_id",
        "common_name":     {"$first": "$common_name"},
        "installed":       {"$first": "$installed"},
        "temporary":       {"$first": "$temporary"},
        "locked":          {"$first": "$locked"},
        "install_date":    {"$first": "$install_date"},
        "terminal_name":   {"$first": "$terminal_name"},
        "removal_date":    {"$first": "$removal_date"},
        "latest_snapshot": {"$first": "$snapshot_date"},
        "snapshot_count":  {"$sum": 1}
    }},

    # 3. Conversão de tipos, normalização de nomes e campos derivados.
    #    Todas as expressões referenciam o estado do documento ANTES desta etapa,
    #    por isso is_active é calculado a partir da string original de removal_date.
    {"$addFields": {
        "place_id": "$_id",

        # Remover espaço-antes-vírgula: primeiro trata duplo-espaço ("  ,"), depois simples (" ,"),
        # depois elimina espaços no início/fim.
        # Ex: "Kennington Road  , Vauxhall" → "Kennington Road, Vauxhall"
        "common_name": {"$trim": {"input": {
            "$replaceAll": {
                "input": {"$replaceAll": {
                    "input": "$common_name",
                    "find": "  ,",
                    "replacement": ","
                }},
                "find": " ,",
                "replacement": ","
            }
        }}},

        # Strings booleanas → booleanos reais
        "installed": {"$cond": [{"$eq": ["$installed", "true"]}, True, False]},
        "temporary":  {"$cond": [{"$eq": ["$temporary",  "true"]}, True, False]},
        "locked":     {"$cond": [{"$eq": ["$locked",     "true"]}, True, False]},

        # String Unix ms → ISODate  (null / vazio / "NULL" → null)
        "install_date": {"$cond": {
            "if": {"$and": [
                {"$ne": ["$install_date", None]},
                {"$ne": ["$install_date", ""]},
                {"$ne": ["$install_date", "NULL"]}
            ]},
            "then": {"$toDate": {"$toLong": "$install_date"}},
            "else": None
        }},
        "removal_date": {"$cond": {
            "if": {"$and": [
                {"$ne": ["$removal_date", None]},
                {"$ne": ["$removal_date", ""]},
                {"$ne": ["$removal_date", "NULL"]}
            ]},
            "then": {"$toDate": {"$toLong": "$removal_date"}},
            "else": None
        }},

        # is_active: estação nunca foi desativada (avaliado contra a string original)
        "is_active": {"$cond": {
            "if": {"$or": [
                {"$eq": ["$removal_date", None]},
                {"$eq": ["$removal_date", ""]},
                {"$eq": ["$removal_date", "NULL"]}
            ]},
            "then": True,
            "else": False
        }},

        # String de data do snapshot → ISODate
        "latest_snapshot": {"$dateFromString": {
            "dateString": "$latest_snapshot",
            "format": "%Y-%m-%d",
            "onError": None
        }}
    }},

    # 4. Forma final: remover _id interno do MongoDB (place_id é a chave natural)
    {"$project": {
        "_id": 0,
        "place_id": 1, "common_name": 1, "terminal_name": 1,
        "installed": 1, "temporary": 1, "locked": 1, "is_active": 1,
        "install_date": 1, "removal_date": 1, "latest_snapshot": 1, "snapshot_count": 1
    }},

    # 5. Substituir atomicamente (ou criar) a collection stations_clean
    {"$out": "stations_clean"}
]

stations_raw.aggregate(pipeline)
print("Pipeline executado — stations_clean criada.")

Pipeline executado — stations_clean criada.


In [11]:
stations_clean = db["stations_clean"]

# Índice único em place_id: garante um documento por estação e acelera joins futuros
stations_clean.create_index("place_id", unique=True)

print(f"Índice único em 'place_id' criado.")
print(f"Documentos em stations_clean: {stations_clean.count_documents({}):,}")

Índice único em 'place_id' criado.
Documentos em stations_clean: 812


---
## **4. Enriquecimento com Coordenadas**

A collection `stations_raw` não contém coordenadas geográficas — estas encontram-se nas collections de disponibilidade de bicicletas, já limpas e estandardizadas no `2-DataUnderstanding.ipynb`. As coordenadas são obtidas diretamente do MongoDB (sem conversão para Pandas nem uso de Spark) usando um `$group` por `place_id` em cada collection sazonal. São usadas as quatro collections para maximizar a cobertura de estações.

In [12]:
# Agregar coordenadas canónicas a partir de todas as collections sazonais.
# Usar as quatro estações maximiza a cobertura (ex: estações instaladas fora do verão).
# O primeiro valor encontrado por place_id é mantido — as coords já foram estandardizadas no notebook 2.
pipeline_coords = [
    {"$group": {"_id": "$place_id", "lat": {"$first": "$lat"}, "lon": {"$first": "$lon"}}}
]

coordenadas = {}
for colecao in ["bikes_clean_summer", "bikes_clean_autumn", "bikes_clean_spring", "bikes_clean_winter"]:
    for doc in db[colecao].aggregate(pipeline_coords):
        if doc["_id"] not in coordenadas:
            coordenadas[doc["_id"]] = (doc["lat"], doc["lon"])

print(f"Estações com coordenadas encontradas: {len(coordenadas):,}")

Estações com coordenadas encontradas: 810


In [13]:
ops = [
    UpdateOne({"place_id": place_id}, {"$set": {"lat": lat, "lon": lon}})
    for place_id, (lat, lon) in coordenadas.items()
]

resultado = stations_clean.bulk_write(ops, ordered=False)
print(f"Estações atualizadas com coordenadas: {resultado.modified_count:,}")
print(f"Estações sem coordenadas (ausentes em todos os datasets): {stations_clean.count_documents({'lat': {'$exists': False}}):,}")

Estações atualizadas com coordenadas: 810
Estações sem coordenadas (ausentes em todos os datasets): 2


---
## **5. Validação**

In [14]:
total_clean = stations_clean.count_documents({})
print(f"Documentos em stations_clean: {total_clean:,}")

print("\nDocumento de exemplo (limpo):")
pprint.pprint(stations_clean.find_one())

Documentos em stations_clean: 812

Documento de exemplo (limpo):
{'_id': ObjectId('6a0e70be39c807d72863892f'),
 'common_name': "St. Martin's Street, West End",
 'install_date': datetime.datetime(2010, 7, 23, 14, 51),
 'installed': True,
 'is_active': True,
 'lat': 51.509087,
 'latest_snapshot': datetime.datetime(2023, 9, 21, 0, 0),
 'locked': False,
 'lon': -0.129697,
 'place_id': 'BikePoints_325',
 'removal_date': None,
 'snapshot_count': 486,
 'temporary': False,
 'terminal_name': '001158'}


In [15]:
print("=== Verificação de Tipos ===")
for field in ["installed", "locked", "temporary"]:
    n_str  = stations_clean.count_documents({field: {"$type": "string"}})
    n_bool = stations_clean.count_documents({field: {"$type": "bool"}})
    print(f"  '{field}': bool={n_bool:,}  string={n_str:,}  {'OK' if n_str == 0 else 'AVISO'}")

n_data = stations_clean.count_documents({"install_date": {"$type": "date"}})
n_nulo = stations_clean.count_documents({"install_date": None})
print(f"\n  'install_date': ISODate={n_data:,}  nulo={n_nulo:,}  {'OK' if n_data + n_nulo == total_clean else 'AVISO'}")

=== Verificação de Tipos ===
  'installed': bool=812  string=0  OK
  'locked': bool=812  string=0  OK
  'temporary': bool=812  string=0  OK

  'install_date': ISODate=723  nulo=89  OK


In [16]:
ativas   = stations_clean.count_documents({"is_active": True})
inativas = stations_clean.count_documents({"is_active": False})
print(f"Estações ativas (sem data de remoção): {ativas:,}")
print(f"Estações desativadas:                  {inativas:,}")
print(f"Estações temporárias: {stations_clean.count_documents({'temporary': True}):,}")
print(f"Estações bloqueadas:  {stations_clean.count_documents({'locked': True}):,}")

# Estações ativas sem coordenadas — instaladas fora do período coberto pelos datasets (Abr 2022–Set 2023)
ativas_sem_coords = stations_clean.count_documents({"is_active": True, "lat": {"$exists": False}})
print(f"\nEstações ativas sem coordenadas: {ativas_sem_coords:,}")
if ativas_sem_coords > 0:
    print("Detalhes:")
    for doc in stations_clean.find(
        {"is_active": True, "lat": {"$exists": False}},
        {"place_id": 1, "common_name": 1, "install_date": 1, "_id": 0}
    ):
        pprint.pprint(doc)

Estações ativas (sem data de remoção): 809
Estações desativadas:                  3
Estações temporárias: 3
Estações bloqueadas:  0

Estações ativas sem coordenadas: 2
Detalhes:
{'common_name': 'Clapham South, Clapham South',
 'install_date': datetime.datetime(2013, 12, 6, 16, 32),
 'place_id': 'BikePoints_753'}
{'common_name': 'Disraeli Road, Putney_old',
 'install_date': None,
 'place_id': 'BikePoints_854'}


In [17]:
print("Estações instaladas por ano:")
for doc in stations_clean.aggregate([
    {"$match": {"install_date": {"$ne": None}}},
    {"$group": {"_id": {"$year": "$install_date"}, "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]):
    print(f"  {doc['_id']}: {doc['count']:>4}  {'█' * (doc['count'] // 3)}")

Estações instaladas por ano:
  2010:  350  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  2011:   44  ██████████████
  2012:   93  ███████████████████████████████
  2013:  142  ███████████████████████████████████████████████
  2014:   26  ████████
  2015:    5  █
  2016:   34  ███████████
  2017:    5  █
  2018:   13  ████
  2019:    3  █
  2020:    4  █
  2022:    4  █


In [18]:
ev = stations_clean.count_documents({"common_name": {"$regex": " ,"}})
ee = stations_clean.count_documents({"common_name": {"$regex": "^\\s|\\s$"}})
print(f"Nomes com ' ,' após limpeza:          {ev}  {'OK' if ev == 0 else 'AVISO'}")
print(f"Nomes com espaço no início/fim:       {ee}  {'OK' if ee == 0 else 'AVISO'}")

Nomes com ' ,' após limpeza:          0  OK
Nomes com espaço no início/fim:       0  OK
